# C02 财务衍生宏观因子
这一节的主线，是把“交易数据”继续往前推进，开始回答更像研究员会问的问题：

> 除了价格本身，我们还能从哪些数据里提取有信息含量的特征？

本节会从三类数据来回答这个问题：
1. 财务数据：公司经营和估值
2. 衍生因子：从价格和成交量再加工出来的特征
3. 宏观变量：公司之外的市场环境


In [1]:
START_DATE = "2022-01-01"
END_DATE = "2024-12-31"
UNIVERSE = ["000001.XSHE", "000002.XSHE", "600000.XSHG", "601318.XSHG"]
# 000001.XSHE: 平安银行
# 000002.XSHE: 万科A
# 600000.XSHG: 浦发银行
# 601318.XSHG: 中国平安

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import rqdatac

# 把教育版 license 直接粘贴到 PASSWD
PASSWD = "A9qCyBHTn1oop9PFrEGs0gX_3brK5gWd3U_bl4Xql-1bdZiu5z93MPkoB0-f99GTJYmnDgFawTLm1xypVYxp-uAsGHpN9I4lMlIgKJMxfzFlLdDi_zIsL6Chzv4xjhjLybpw3KlegEPco27BZ5it8gCb53opmnRv1xvM937m0f0=HAf_ZY_gevO08quesn6NLrx9ygL3WCtsoFICUIfRNnVXpzM3a288oZNyWvlt5VahwuPW1Y8MTQflW3XoR9e__1nZARS8VzPlEYcr9tMz7erex--jctwuQYoZDd8CGy5VmMa6bvFKv-kK0dhsbB9M9hK9LKdD3k1opU0JqYrPW7c="

if PASSWD:
    rqdatac.init('license', PASSWD)
    print("rqdatac 初始化成功")
else:
    print("请先填写 PASSWD，再运行本单元")


rqdatac 初始化成功


d:\python_workspace\Quant\.venv\liangwen2\Lib\site-packages\rqdatac\client.py:263: UserWarning: Your account will be expired after  467 days. Please call us at 0755-22676337 to upgrade or purchase or renew your contract.
  warnings.warn("Your account will be expired after  {} days. "


## 1) 课程目标
这一节结束后，希望大家至少能建立下面这条链路：

1. 知道 `get_pit_financials_ex` 和 `get_factor` 各自适合干什么
2. 知道 `ttm / lyr / mrq` 这三个口径分别在回答什么问题
3. 知道衍生因子不神秘，本质还是对价格序列再加工
4. 知道宏观变量可以作为市场状态的补充解释
5. 会把多来源数据拼成一张最小可用的 `factor_panel`


## 2) 为什么财务数据值得单独讲
在第一节课里，我们拿到的是：`close`、`prev_close`、`volume`。这些字段回答的是“市场今天怎么走”。

但如果我们想研究“什么样的公司更值得长期持有”，就会自然开始关心：
- 收入增长是否稳定
- 利润质量是否可靠
- 资产负债结构是否健康
- 当前估值是不是太贵

这时光看交易数据就不够了，必须引入财务信息。


### 2.1 `get_pit_financials_ex` 和 `get_factor` 的区别
这两个接口都和“财务”有关，但角色不一样：

1. `get_pit_financials_ex`
- 更像原始报表字段
- 适合讲清楚字段本身的经济含义
- 更接近“财报原材料”

2. `get_factor`
- 更像加工好的研究特征
- 适合后面做因子分析和策略构建
- 更接近“研究成品或半成品”

一句话记：
- 想理解报表字段本身，用 `get_pit_financials_ex`
- 想快速拿研究特征，用 `get_factor`


In [ ]:
# 先看原始财务字段：
# 这里选两只熟悉的股票，方便课堂上并排对比
fin_raw = rqdatac.get_pit_financials_ex(
    fields=["revenue", "net_profit", "total_assets", "total_equity"],
    order_book_ids=["000001.XSHE", "000002.XSHE"],
    start_quarter="2022q1",
    end_quarter="2024q4",
    statements="all",
)

# reset_index() 的目的和 C01 一样：把索引信息变成普通列，方便观察和 merge
fin_raw = fin_raw.reset_index()
fin_raw.head()


### 2.2 `ttm / lyr / mrq` 是什么
这是量化里经常一起出现的三个口径：

1. `ttm` = trailing twelve months
- 过去 12 个月滚动值
- 常用于利润表类指标，例如收入、净利润

2. `lyr` = last year report
- 上一个完整年度的年报值
- 更适合看稳定的年度口径

3. `mrq` = most recent quarter
- 最近一个季度值
- 更适合看季度边际变化

真正要记住的不是缩写，而是：
**同一个字段，不同口径回答的问题不同。**


In [ ]:
# 用 get_factor 看同类财务信息在不同口径下的表现
factor_compare = rqdatac.get_factor(
    ["000001.XSHE", "000002.XSHE"],
    [
        "pe_ratio_ttm",
        "total_income_ttm_0",
        "total_income_lyr_0",
        "total_income_mrq_0",
    ],
    start_date=START_DATE,
    end_date=END_DATE,
    expect_df=True,
)

factor_compare = factor_compare.reset_index()
factor_compare.head()


### 2.3 为什么这里会牵扯“未来函数”
财务数据不是每天都更新，而是跟着财报披露节奏来的。

所以研究时必须注意：
- 某一天能看到的数据，必须是那天之前已经披露的
- 不能把未来财报数据拿回过去使用

这也是为什么财务数据比交易数据更容易出“看起来很准、其实违规”的结果。


In [ ]:
# 选一只股票，画出不同财务口径随时间的变化
one_stock = factor_compare.loc[
    factor_compare["order_book_id"] == "000001.XSHE"
].copy()

fig, ax = plt.subplots(figsize=(10, 4))
one_stock.plot(x="date", y="total_income_ttm_0", ax=ax, label="income_ttm")
one_stock.plot(x="date", y="total_income_lyr_0", ax=ax, label="income_lyr")
one_stock.plot(x="date", y="total_income_mrq_0", ax=ax, label="income_mrq")
ax.set_title("同一只股票在不同财务口径下的收入指标")
plt.show()


### 2.4 这张图到底在看什么
这张图不是为了比较哪条线更好看，而是帮助学生建立两个直觉：

1. 财务字段不是只有一个“唯一正确数字”
- 同一个概念，可能有滚动口径、年报口径、季度口径

2. 研究时必须先想清楚自己要回答什么问题
- 如果你关心长期经营能力，可能更偏向 `ttm / lyr`
- 如果你关心最近经营变化，可能更偏向 `mrq`


## 3) 衍生因子：价格数据进一步加工后能得到什么
上一节课里我们已经接触过：
- `shift()`
- `rolling()`
- 波动率
- 均线

衍生因子其实就是把这种思路继续推进：
- 趋势类
- 波动类
- 量价类
- 技术指标类

上一届课件里提到过 `Alpha101`，这里保留一个演示版本，不展开全部细节。


In [ ]:
# 先直接读取平台已经加工好的衍生因子
alpha_demo = rqdatac.get_factor(
    ["000001.XSHE", "600000.XSHG"],
    ["WorldQuant_alpha001", "WorldQuant_alpha005", "WorldQuant_alpha010"],
    start_date=START_DATE,
    end_date=END_DATE,
    expect_df=True,
)

alpha_demo = alpha_demo.reset_index()
alpha_demo.head()


In [ ]:
# 再自己从价格序列加工出两个简单特征：20日动量和20日波动率
price_demo = rqdatac.get_price(
    ["000001.XSHE", "600000.XSHG"],
    start_date=START_DATE,
    end_date=END_DATE,
    fields=["close"],
    adjust_type="pre",
    expect_df=True,
).reset_index()

price_demo = price_demo.sort_values(["order_book_id", "date"]).copy()
price_demo["ret"] = price_demo["close"] / price_demo.groupby("order_book_id")["close"].shift(1) - 1
price_demo["mom_20"] = price_demo["close"] / price_demo.groupby("order_book_id")["close"].shift(20) - 1
price_demo["vol_20"] = (
    price_demo.groupby("order_book_id")["ret"]
    .rolling(20)
    .std()
    .reset_index(level=0, drop=True)
)

price_demo.head()


### 3.1 为什么说衍生因子并不神秘
因为从代码上看，很多所谓“因子”其实只是：
- 对已有数据做移动窗口统计
- 对收益做累计或标准化
- 对量价关系做再表达

真正重要的不是背公式，而是知道自己在从哪一类数据里提取什么信息。


## 4) 宏观数据：公司之外的外部环境
如果只看个股层面的信息，我们还是少了一块拼图：
- 利率环境
- 流动性环境
- 经济周期

这些变量不一定直接告诉你“下一只股票买谁”，但常常能帮助我们理解市场处在什么状态。


In [ ]:
# 演示两条常见宏观序列
rr = rqdatac.econ.get_macro("china_reserve_requirement_ratio")
m2 = rqdatac.econ.get_macro("china_money_supply_m2")

rr = rr.loc[(rr.index >= pd.Timestamp(START_DATE)) & (rr.index <= pd.Timestamp(END_DATE))]
m2 = m2.loc[(m2.index >= pd.Timestamp(START_DATE)) & (m2.index <= pd.Timestamp(END_DATE))]

fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
rr.plot(ax=axes[0], title="存款准备金率")
m2.plot(ax=axes[1], title="M2")
plt.tight_layout()
plt.show()


### 4.1 宏观数据在这门课里的定位
这门 workshop 不会把宏观做得特别深，所以你可以把它理解成：
- 它不是这一节的主角
- 但它提醒大家，因子研究不只有公司和价格，还可以有更宏观的背景变量


## 5) 把多来源数据拼成一张因子面板
真正进入多因子研究时，我们不会只看一个字段，而是会把多类信息拼成一张统一的表。

这张表最少应该包含：
- `date`
- `order_book_id`
- 若干个候选因子

后面做标准化、IC 分析、回测，都是从这张表继续往下走。


In [ ]:
# 这里做一张最小版本的因子面板
basic_factor = rqdatac.get_factor(
    ["000001.XSHE", "600000.XSHG"],
    ["pe_ratio_ttm"],
    start_date=START_DATE,
    end_date=END_DATE,
    expect_df=True,
).reset_index()

factor_panel = price_demo.merge(
    basic_factor,
    on=["date", "order_book_id"],
    how="left",
)[["date", "order_book_id", "mom_20", "vol_20", "pe_ratio_ttm"]].dropna()

factor_panel.head()


### 5.1 为什么这张 `factor_panel` 很关键
你可以把它理解成后面整套多因子流程的“中间层接口”：
- 上游是各种原始数据和衍生数据
- 下游是标准化、IC、回测和组合构建

如果这张表结构不清楚，后面每一步都会变得混乱。


## 6) 课上小练习


### 练习 1：自己读取一个估值因子
要求：
1. 用 `rqdatac.get_factor` 读取 `000001.XSHE` 的 `pb_ratio_ttm`
2. 时间区间使用本节的 `START_DATE` 到 `END_DATE`
3. `reset_index()` 后展示前 5 行


In [ ]:
# 练习 1：学生现场自己写
# 提示：get_factor -> reset_index -> head


In [ ]:
# 参考答案
ex1_factor = rqdatac.get_factor(
    ["000001.XSHE"],
    ["pb_ratio_ttm"],
    start_date=START_DATE,
    end_date=END_DATE,
    expect_df=True,
).reset_index()

ex1_factor.head()


### 练习 2：自己拼一个两因子面板
要求：
1. 从 `price_demo` 中保留 `date`、`order_book_id`、`mom_20`
2. 再用 `rqdatac.get_factor` 读取 `pb_ratio_ttm`
3. 用 `merge` 把两张表拼起来


In [ ]:
# 练习 2：学生现场自己写
# 提示：merge 的键仍然是 date 和 order_book_id


In [ ]:
# 参考答案
ex2_price = price_demo[["date", "order_book_id", "mom_20"]].copy()
ex2_value = rqdatac.get_factor(
    ["000001.XSHE", "600000.XSHG"],
    ["pb_ratio_ttm"],
    start_date=START_DATE,
    end_date=END_DATE,
    expect_df=True,
).reset_index()

ex2_panel = ex2_price.merge(
    ex2_value,
    on=["date", "order_book_id"],
    how="left",
)

ex2_panel.head()


## 小结
这一节的关键词是：**因子原料**。

本节结束后，大家应该建立起下面这条链路：
1. 交易数据告诉我们“价格怎么走”
2. 财务数据告诉我们“公司基本面如何”
3. 衍生因子把价格进一步加工成研究特征
4. 宏观数据补充市场外部环境
5. 最终这些信息会被拼成一张 `factor_panel`

本节常见易错点：
1. 把 `ttm / lyr / mrq` 混在一起，不问自己到底在看什么口径
2. 忽略财务数据的披露时点，误用未来信息
3. 只会取单个字段，不会把多来源数据组织成统一面板

下一节我们会把这些信号真正接进回测框架，回答“如果按规则交易，净值会怎样变化”。
